#Schritt 1b: Benchmark-Datensatz laden

Bereits annotierten Lyrics-Datensatz nach Russells Vier-Quadranten-Modell: http://mir.dei.uc.pt/downloads.html

Referenz: Malheiro et al. (2018), IEEE TAFFC 9(2), 240–254.

Der Datensatz besteht aus zwei Teilen: 180 Songs mit kontinuierlichen Valence-/Arousal-Werten und 771 Songs mit einem fertigen Quadranten-Label.

In [ ]:
import pandas as pd
import os
PFAD_VALENCE_AROUSAL = 'data/MER_lyrics_dataset.xls'
PFAD_QUADRANT = 'data/Dataset-AllMusic-771Lyrics.xls'

In [ ]:
def lade_valence_arousal_datensatz(pfad=PFAD_VALENCE_AROUSAL):
    daten = pd.read_excel(pfad, sheet_name='Folha1', skiprows=18)
    daten = daten.rename(columns={'Song ': 'title', 'Artist': 'artist', 'Code': 'song_id', 'Avg_Valence': 'valence', 'Avg_Arousal': 'arousal', 'Source of the Lyric ': 'quelle_url'})

    def bestimme_quadrant(zeile):
        # Russells Modell teilt den Raum am Nullpunkt der beiden Achsen:
        # Valence = Stimmungsrichtung (positiv/negativ), Arousal = Aktivierung (aufgeregt/ruhig).
        v_positiv = zeile['valence'] >= 0
        a_positiv = zeile['arousal'] >= 0
        if v_positiv and a_positiv:
            return 'Q1'
        elif not v_positiv and a_positiv:
            return 'Q2'
        elif not v_positiv and (not a_positiv):
            return 'Q3'
        else:
            return 'Q4'
    daten['quadrant'] = daten.apply(bestimme_quadrant, axis=1)
    daten['quelle'] = 'MER_180_valence_arousal'
    return daten[['song_id', 'title', 'artist', 'valence', 'arousal', 'quadrant', 'quelle']]

In [ ]:
def lade_quadrant_datensatz(pfad=PFAD_QUADRANT):
    daten = pd.read_excel(pfad, sheet_name='Folha1', skiprows=14)
    daten = daten.rename(columns={'Name': 'song_id', 'Class': 'quadrant', 'Artist': 'artist', 'Title': 'title', 'Unnamed: 4': 'quelle_url'})
    daten['artist'] = daten['artist'].str.replace('_', ' ', regex=False)
    daten['title'] = daten['title'].str.replace('_', ' ', regex=False)
    daten['valence'] = None
    daten['arousal'] = None
    daten['quelle'] = 'AllMusic_771_quadrant'
    return daten[['song_id', 'title', 'artist', 'valence', 'arousal', 'quadrant', 'quelle']]

In [ ]:
def quadrant_zu_label(quadrant): 
    zuordnung = {'Q1': 'eher_unauffällig', 'Q2': 'eher_unauffällig', 'Q3': 'moeglicherweise_auffällig', 'Q4': 'eher_unauffällig'}
    return zuordnung.get(quadrant, 'unbekannt')

In [ ]:
def lyrics_für_benchmark(benchmark_labels, genius_token, anzahl=30, zufalls_seed=42, timeout=15, retries=3):
    import lyricsgenius
    genius = lyricsgenius.Genius(genius_token, timeout=timeout, retries=retries)
    genius.remove_section_headers = False
    stichprobe = benchmark_labels.sample(n=min(anzahl, len(benchmark_labels)), random_state=zufalls_seed).copy()
    lyrics_liste = []
    fehlgeschlagen = []
    for index, zeile in stichprobe.iterrows():
        titel = zeile['title']
        interpret = zeile['artist']
        print(f'Suche: {titel} von {interpret} ...')
        try:
            song = genius.search_song(titel, interpret)
        except Exception as fehler:
            print(f'Fehler ({type(fehler).__name__}: {fehler}), wird übersprungen')
            fehlgeschlagen.append((interpret, titel, str(fehler)))
            lyrics_liste.append(None)
            continue
        if song is not None:
            lyrics_liste.append(song.lyrics)
        else:
            print('nicht gefunden, wird übersprungen')
            fehlgeschlagen.append((interpret, titel, 'nicht gefunden'))
            lyrics_liste.append(None)
    stichprobe['lyrics'] = lyrics_liste
    # Songs, für die Genius nichts liefert, fliegen aus der Stichprobe raus, sonst geht der mit leerem Text bei 1c als neutral durch
    stichprobe = stichprobe[stichprobe['lyrics'].notna()].reset_index(drop=True)
    print(f'\n{len(stichprobe)} von {anzahl} Songs erfolgreich mit Lyrics geladen.')
    if fehlgeschlagen:
        print(f'{len(fehlgeschlagen)} übersprungen:')
        for a, t, g in fehlgeschlagen:
            print(f'  - {t} ({a}): {g}')
    return stichprobe

#Pipeline ausführen

In [ ]:
va_labels = lade_valence_arousal_datensatz()
quadrant_labels = lade_quadrant_datensatz()
alle_labels = pd.concat([va_labels, quadrant_labels], ignore_index=True)
alle_labels['benchmark_grob'] = alle_labels['quadrant'].apply(quadrant_zu_label)
print(f'Insgesamt {len(alle_labels)} Songs mit Labels geladen ')
print(f'({len(va_labels)} aus MER_180 + {len(quadrant_labels)} aus AllMusic_771).')
print(alle_labels['quadrant'].value_counts())
alle_labels.to_csv('data/benchmark_labels_alle.csv', index=False)
print('\nAlle Labels (ohne Lyrics) gespeichert unter: data/benchmark_labels_alle.csv')

In [ ]:
mein_token = 'HIER_GENIUS_TOKEN_EINFUEGEN'
if mein_token == 'HIER_GENIUS_TOKEN_EINFUEGEN':
    raise ValueError('Bitte oben Genius-Token eintragen.')

In [ ]:
benchmark_mit_lyrics = lyrics_für_benchmark(alle_labels, mein_token, anzahl=30)
benchmark_mit_lyrics.to_csv('data/benchmark_lyrics_mit_labels.csv', index=False)
print('\nGespeichert unter: data/benchmark_lyrics_mit_labels.csv')